In [1]:
# %%
# Import basic libraries
import pandas as pd
import numpy as np
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import os 
from pathlib import Path
import sys
from tqdm import tqdm
import pycwt as wavelet
from pycwt import wct_significance
from pycwt import helpers
import matplotlib.dates as mdates
import scipy
from scipy.signal import detrend
from matplotlib.ticker import LogLocator, FormatStrFormatter

# Read Geoparquet file
gaul_gdf = gpd.read_parquet("/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/external/geoparquet/GAUL_2024_L2.parquet")
ne_admin_0_gdf = gpd.read_parquet("/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/external/geoparquet/ne_10m_admin_0_countries.parquet")
ne_admin_1_gdf = gpd.read_parquet("/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/external/geoparquet/ne_10m_admin_1_states_provinces.parquet")

import dask.dataframe as dd

# ---------------------------
# 1️⃣ Load CSV with explicit dtypes
# ---------------------------
dtype_dict = {
    'adm_0_name': 'object',
    'adm_1_name': 'object',
    'adm_2_name': 'object',
    'dengue_total': 'float64'
}

national_ddf = dd.read_csv(
    "/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/raw/global/National_extract_V1_3.csv",
    dtype=dtype_dict
)
spatial_ddf = dd.read_csv(
    "/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/raw/global/Spatial_extract_V1_3.csv",
    dtype=dtype_dict
)

temporal_ddf = dd.read_csv(
    "/home/patwuch/Documents/projects/Chuang-Lab-TMU/dengue-infection-module/main/raw/global/Temporal_extract_V1_3.csv",
    dtype=dtype_dict
)


In [ ]:
national_ddf.columns

In [4]:
gaul_gdf

,iso3_code,map_code,gaul0_code,gaul0_name,gaul1_code,gaul1_name,gaul2_code,gaul2_name,continent,disp_en,geometry
0,TZA,TZA,166,United Republic of Tanzania,1679,Arusha,106380,Arumeru,Africa,"Arumeru, Arusha, United Republic of Tanzania","POLYGON ((36.71467 -3.56221, 36.71439 -3.56391..."
1,TZA,TZA,166,United Republic of Tanzania,1679,Arusha,106381,Arusha,Africa,"Arusha, Arusha, United Republic of Tanzania","POLYGON ((36.71475 -3.5607, 36.70735 -3.56379,..."
2,TZA,TZA,166,United Republic of Tanzania,1679,Arusha,106382,Karatu,Africa,"Karatu, Arusha, United Republic of Tanzania","POLYGON ((35.83594 -3.36915, 35.83334 -3.37666..."
3,TZA,TZA,166,United Republic of Tanzania,1679,Arusha,106383,Longido,Africa,"Longido, Arusha, United Republic of Tanzania","POLYGON ((36.17068 -2.20099, 36.1709 -2.20112,..."
4,TZA,TZA,166,United Republic of Tanzania,1679,Arusha,106384,Monduli,Africa,"Monduli, Arusha, United Republic of Tanzania","POLYGON ((36.60573 -3.10696, 36.6485 -3.14393,..."
...,...,...,...,...,...,...,...,...,...,...,...
45519,SLV,SLV,191,El Salvador,2025,Sonsonate,145717,Sonsonate Oeste,America,"Sonsonate Oeste, Sonsonate, El Salvador","POLYGON ((-89.81993 13.66957, -89.81818 13.668..."
45520,SLV,SLV,191,El Salvador,2026,Usulután,145718,Usulután Este,America,"Usulután Este, Usulután, El Salvador","MULTIPOLYGON (((-88.46236 13.20625, -88.46236 ..."
45521,SLV,SLV,191,El Salvador,2026,Usulután,145719,Usulután Norte,America,"Usulután Norte, Usulután, El Salvador","POLYGON ((-88.50626 13.45661, -88.50713 13.455..."
45522,SLV,SLV,191,El Salvador,2026,Usulután,145720,Usulután Oeste,America,"Usulután Oeste, Usulután, El Salvador","MULTIPOLYGON (((-88.46264 13.17875, -88.46236 ..."


In [7]:
# Assuming your national table is df_national with RNE_iso_code
matches = {}
for col in ne_admin_0_gdf.columns:
    if ne_admin_0_gdf[col].dtype == object:  # string-like
        match_ratio = (ne_admin_0_gdf[col].isin(national_ddf['RNE_iso_code'])).mean()
        matches[col] = match_ratio

# Sort by highest match
matches_sorted = sorted(matches.items(), key=lambda x: x[1], reverse=True)
print(matches_sorted)


[('ADM0_A3_CN', np.float64(0.5038759689922481)), ('ADM0_A3_AR', np.float64(0.5038759689922481)), ('ISO_A3_EH', np.float64(0.5)), ('ADM0_A3_RU', np.float64(0.5)), ('ADM0_A3_TW', np.float64(0.5)), ('ADM0_A3_BR', np.float64(0.5)), ('ADM0_A3_VN', np.float64(0.5)), ('ADM0_A3_FR', np.float64(0.49612403100775193)), ('ADM0_A3_ES', np.float64(0.49612403100775193)), ('ADM0_A3_IN', np.float64(0.49612403100775193)), ('ADM0_A3_NP', np.float64(0.49612403100775193)), ('ADM0_A3_PK', np.float64(0.49612403100775193)), ('ADM0_A3_DE', np.float64(0.49612403100775193)), ('ADM0_A3_GB', np.float64(0.49612403100775193)), ('ADM0_A3_IL', np.float64(0.49612403100775193)), ('ADM0_A3_PS', np.float64(0.49612403100775193)), ('ADM0_A3_SA', np.float64(0.49612403100775193)), ('ADM0_A3_EG', np.float64(0.49612403100775193)), ('ADM0_A3_MA', np.float64(0.49612403100775193)), ('ADM0_A3_PT', np.float64(0.49612403100775193)), ('ADM0_A3_JP', np.float64(0.49612403100775193)), ('ADM0_A3_KO', np.float64(0.49612403100775193)), ('AD

In [8]:
ne_admin_0_gdf['ADM0_A3_CN'].dtype


dtype('O')

In [10]:
set(national_ddf['RNE_iso_code']) - set(ne_admin_0_gdf['ADM0_A3_CN'])


{'TWN'}

In [14]:
unique_ne_iso = set(ne_admin_0_gdf['ADM0_A3_CN'].unique())
unique_national_iso = set(national_ddf['RNE_iso_code'].unique())

match_fraction = len(unique_national_iso & unique_ne_iso) / len(unique_national_iso)
print(match_fraction)


0.992


In [19]:
unique_ne_iso = set(ne_admin_0_gdf['ISO_A3'].unique())
unique_national_iso = set(national_ddf['RNE_iso_code'].unique())

missing_countries = unique_national_iso - unique_ne_iso
print(missing_countries)


{'FRA', 'CLP'}


In [20]:
# Show ISO_A3 values in NE for France and Chile
ne_admin_0_gdf[ne_admin_0_gdf['ADM0_A3_CN'].isin(['FRA', 'CLP'])][['NAME', 'ISO_A3', 'ADM0_A3_CN']]


,NAME,ISO_A3,ADM0_A3_CN
21,France,-99,FRA
252,Clipperton I.,-99,CLP


In [23]:
iso_mapping = {
    'FRA': 'FR',   # France mainland
    'CLP': 'CLP',  # assign its ADM0_A3_CN code manually
    'TWN': 'TWN'   # Taiwan stays separate
}
national_ddf['ISO_A3_join'] = national_ddf['RNE_iso_code'].replace(iso_mapping)


In [26]:
import pandas as pd
import plotly.express as px

# Map textual resolutions to numbers
res_mapping = {'Year': 1, 'Month': 2, 'Week': 3}
national_ddf['T_res_num'] = national_ddf['T_res'].map(res_mapping)

# Make sure your ISO codes match the GeoParquet
national_ddf['ISO_A3_join'] = national_ddf['RNE_iso_code'].replace({
    'FRA': 'FR',
    'CLP': 'CLP',
    'TWN': 'TWN'
})


/home/patwuch/miniforge3/envs/gee_dengue_ml/lib/python3.10/site-packages/dask/dataframe/dask_expr/_collection.py:4218: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('T_res', 'float64'))

  warnings.warn(meta_warning(meta, method="map"))


In [27]:
agg_df = national_ddf.groupby(['ISO_A3_join', 'Year']).agg({
    'T_res_num': 'max'  # highest available resolution
}).reset_index()


In [ ]:
import dask.dataframe as dd
import geopandas as gpd
import plotly.express as px

# ---------------------------
# 1️⃣ Load / map ISO codes and resolution in Dask
# ---------------------------
# Assuming national_ddf is a Dask DataFrame

# Map ISO codes
iso_mapping = {'FRA': 'FR', 'CLP': 'CLP', 'TWN': 'TWN'}
national_ddf['ISO_A3_join'] = national_ddf['RNE_iso_code'].replace(iso_mapping)

# Map T_res to numeric
res_mapping = {'Year': 1, 'Month': 2, 'Week': 3}
national_ddf['T_res_num'] = national_ddf['T_res'].map(res_mapping)

# ---------------------------
# 2️⃣ Aggregate per country per year (Dask can do this lazily)
# ---------------------------
agg_ddf = national_ddf.groupby(['ISO_A3_join', 'Year'])['T_res_num'].max().reset_index()

# ---------------------------
# 3️⃣ Compute the aggregated DataFrame (small enough to fit in memory)
# ---------------------------
agg_df = agg_ddf.compute()  # now a pandas DataFrame

# ---------------------------
# 4️⃣ Merge with your GeoParquet
# ---------------------------
ne_admin_0_gdf['ISO_A3'] = ne_admin_0_gdf['ISO_A3'].astype(str)
agg_df['ISO_A3_join'] = agg_df['ISO_A3_join'].astype(str)

merged_gdf = ne_admin_0_gdf.merge(
    agg_df,
    left_on='ISO_A3',
    right_on='ISO_A3_join',
    how='left'
)

# ---------------------------
# 5️⃣ Make the animated choropleth
# ---------------------------
fig = px.choropleth(
    merged_gdf,
    geojson=merged_gdf.geometry,
    locations=merged_gdf.index,
    color='T_res_num',
    hover_name='NAME',
    animation_frame='Year',
    color_continuous_scale='YlOrRd',
    range_color=[0, 3],
    labels={'T_res_num': 'Time Resolution'}
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    title_text='Global Infection Data Availability & Resolution Over Years',
    coloraxis_colorbar=dict(
        tickvals=[1, 2, 3],
        ticktext=['Year', 'Month', 'Week']
    )
)
fig.show()


/home/patwuch/miniforge3/envs/gee_dengue_ml/lib/python3.10/site-packages/dask/dataframe/dask_expr/_collection.py:4218: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('T_res', 'float64'))

  warnings.warn(meta_warning(meta, method="map"))


In [18]:
ne_admin_0_gdf.columns.to_list()

['featurecla',
 'scalerank',
 'LABELRANK',
 'SOVEREIGNT',
 'SOV_A3',
 'ADM0_DIF',
 'LEVEL',
 'TYPE',
 'TLC',
 'ADMIN',
 'ADM0_A3',
 'GEOU_DIF',
 'GEOUNIT',
 'GU_A3',
 'SU_DIF',
 'SUBUNIT',
 'SU_A3',
 'BRK_DIFF',
 'NAME',
 'NAME_LONG',
 'BRK_A3',
 'BRK_NAME',
 'BRK_GROUP',
 'ABBREV',
 'POSTAL',
 'FORMAL_EN',
 'FORMAL_FR',
 'NAME_CIAWF',
 'NOTE_ADM0',
 'NOTE_BRK',
 'NAME_SORT',
 'NAME_ALT',
 'MAPCOLOR7',
 'MAPCOLOR8',
 'MAPCOLOR9',
 'MAPCOLOR13',
 'POP_EST',
 'POP_RANK',
 'POP_YEAR',
 'GDP_MD',
 'GDP_YEAR',
 'ECONOMY',
 'INCOME_GRP',
 'FIPS_10',
 'ISO_A2',
 'ISO_A2_EH',
 'ISO_A3',
 'ISO_A3_EH',
 'ISO_N3',
 'ISO_N3_EH',
 'UN_A3',
 'WB_A2',
 'WB_A3',
 'WOE_ID',
 'WOE_ID_EH',
 'WOE_NOTE',
 'ADM0_ISO',
 'ADM0_DIFF',
 'ADM0_TLC',
 'ADM0_A3_US',
 'ADM0_A3_FR',
 'ADM0_A3_RU',
 'ADM0_A3_ES',
 'ADM0_A3_CN',
 'ADM0_A3_TW',
 'ADM0_A3_IN',
 'ADM0_A3_NP',
 'ADM0_A3_PK',
 'ADM0_A3_DE',
 'ADM0_A3_GB',
 'ADM0_A3_BR',
 'ADM0_A3_IL',
 'ADM0_A3_PS',
 'ADM0_A3_SA',
 'ADM0_A3_EG',
 'ADM0_A3_MA',
 'ADM0_A3_PT

In [ ]:
national_ddf = national_ddf.rename(columns={
    'adm_0_name': 'gaul0_name'})

temporal_ddf = temporal_ddf.rename(columns={
    'adm_0_name': 'gaul0_name',
    'adm_1_name': 'gaul1_name',
    'adm_2_name': 'gaul2_name'
})

spatial_ddf = spatial_ddf.rename(columns={
    'adm_0_name': 'gaul0_name',
    'adm_1_name': 'gaul1_name',
    'adm_2_name': 'gaul2_name'
})

In [ ]:
print("Unique values in GAUL geoparquet (gaul0_name, gaul1_name, gaul2_name):")
print(gaul_gdf[['gaul0_name', 'gaul1_name', 'gaul2_name']].nunique())

# National only has gaul0_name
print("Unique values in national_ddf (gaul0_name):")
print(national_ddf['gaul0_name'].nunique().compute())

# Temporal has gaul0_name, gaul1_name, gaul2_name
print("\nUnique values in temporal_ddf:")
print(temporal_ddf[['gaul0_name', 'gaul1_name', 'gaul2_name']].nunique().compute())

# Spatial has gaul0_name, gaul1_name, gaul2_name
print("\nUnique values in spatial_ddf:")
print(spatial_ddf[['gaul0_name', 'gaul1_name', 'gaul2_name']].nunique().compute())



In [ ]:
gdf_l0 = gaul_gdf['gaul0_name'].unique().tolist()
gdf_l1 = gaul_gdf['gaul1_name'].unique().tolist()
gdf_l2 = gaul_gdf['gaul2_name'].unique().tolist()

N_l0 = national_ddf['gaul0_name'].unique().compute().tolist()
T_l0 = temporal_ddf['gaul0_name'].unique().compute().tolist()
T_l1 = temporal_ddf['gaul1_name'].unique().compute().tolist()
T_l2 = temporal_ddf['gaul2_name'].unique().compute().tolist()
S_l0 = spatial_ddf['gaul0_name'].unique().compute().tolist()
S_l1 = spatial_ddf['gaul1_name'].unique().compute().tolist()
S_l2 = spatial_ddf['gaul2_name'].unique().compute().tolist()


In [ ]:
N_merge = national_ddf.merge(
    gaul_gdf[['gaul0_name']].drop_duplicates(),
    on='gaul0_name',
    how='left',
    indicator=True
).compute()

# Rows in national_ddf that didn't match
N_mismatches = N_merge[N_merge['_merge'] == 'left_only']

# See how many there are
print("Number of mismatched rows:", len(N_mismatches))


# Get unique gaul0_names that didn't match
unmatched_names = N_merge.loc[N_merge['_merge'] == 'left_only', 'gaul0_name'].unique()

print("GAUL0 names with mismatches:")
print(unmatched_names)


In [ ]:
# Rows in national_ddf that didn't match
N_mismatches = N_merge[N_merge['_merge'] == 'left_only']

# See how many there are
print("Number of mismatched rows:", len(N_mismatches))

# Optionally, inspect them
print(N_mismatches.sample())


In [ ]:
# Limit the number of rows to 10
pd.set_option('display.max_rows', 10)

# Limit the number of columns to 5
pd.set_option('display.max_columns', 5)

# Limit the column width to 30 characters
pd.set_option('display.max_colwidth', 30)

In [ ]:
from rapidfuzz import process, fuzz
import pandas as pd

# 1) Missing names from the merge (N_merge is already Pandas after .compute())
N_missing = N_merge.loc[N_merge['_merge'] == 'left_only', 'gaul0_name'].unique().tolist()

# 2) Unique names in gaul_gdf (GeoPandas / Pandas)

# 3) Unique names in national_ddf (Dask → compute first)
ddf_names = national_ddf['gaul0_name'].unique().compute().tolist()

results = []

for name in missing:
    in_ddf = name in ddf_names
    
    # Fuzzy match in gdf
    match = process.extractOne(
        name,
        gdf_names,
        scorer=fuzz.token_sort_ratio
    )
    
    if match:
        best_match_name, score, idx = match
        if score >= 60:
            in_gdf = True
        else:
            best_match_name = None
            in_gdf = False
    else:
        best_match_name = None
        in_gdf = False

    results.append({
        'name_in_ddf': name,
        'best_match_in_gdf': best_match_name,
        'score': score if best_match_name else None,
        'exists_in_ddf': in_ddf,
        'exists_in_gdf': in_gdf
    })

summary_df = pd.DataFrame(results)
print(summary_df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RES_PRIORITY = ["Week", "Month", "Year"]  # higher resolution first
RES_COLOR_MAP = {"Week": "blue", "Month": "green", "Year": "orange"}

def plot_all_countries(df, output_dir, start_date="2000-01-01"):
    df = df.copy()
    start_date = pd.to_datetime(start_date)

    # Ensure date columns are datetime
    for col in ["calendar_start_date", "calendar_end_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    countries = df["full_name"].dropna().unique()

    for country in countries:
        country_df = df[df["full_name"] == country].copy()
        country_df = country_df.dropna(subset=["calendar_start_date", "calendar_end_date"])
        if country_df.empty or country_df["calendar_end_date"].max() < start_date:
            continue

        # Clip start date
        country_df = country_df[country_df["calendar_end_date"] >= start_date]

        # Expand to daily pseudo-cases with T_res
        country_df["start_clipped"] = country_df["calendar_start_date"].apply(lambda x: max(x, start_date))
        country_df["date"] = country_df.apply(
            lambda r: pd.date_range(r["start_clipped"], r["calendar_end_date"], freq="D")
            if r["calendar_end_date"] >= r["start_clipped"] else pd.DatetimeIndex([]),
            axis=1
        )
        country_df["n_days"] = country_df["date"].str.len()
        country_df["daily_cases"] = country_df["dengue_total"] / country_df["n_days"]

        # Explode
        daily_df = country_df[["date", "daily_cases", "T_res"]].explode("date")
        if daily_df.empty:
            continue

        # Normalize T_res
        daily_df["T_res"] = daily_df["T_res"].astype(str).str.lower().str.strip()

        # -------------------------
        # For each date, pick highest resolution available
        # -------------------------
        daily_df["res_priority"] = daily_df["T_res"].apply(lambda x: RES_PRIORITY.index(x) if x in RES_PRIORITY else len(RES_PRIORITY))
        daily_df = daily_df.sort_values(["date", "res_priority"])
        daily_df = daily_df.drop_duplicates("date", keep="first")
        daily_df = daily_df.sort_values("date")  # final daily_df is continuous

        # Check for overlapping resolutions in the raw data
        raw_overlaps = daily_df.groupby("date")["T_res"].nunique()
        overlap_dates = raw_overlaps[raw_overlaps > 1].index
        if len(overlap_dates) > 0:
            print(f"WARNING: {len(overlap_dates)} dates had multiple resolutions for {country}. First 10: {list(overlap_dates[:10])}")

        # Compute cumulative cases
        daily_df["cumulative_cases"] = daily_df["daily_cases"].cumsum()

        # -------------------------
        # Helper to plot continuous line segments colored by resolution
        # -------------------------
        def plot_colored_line(ax, x, y, res_series):
            start_idx = 0
            while start_idx < len(x):
                current_res = res_series.iloc[start_idx]
                # find contiguous segment with same resolution
                end_idx = start_idx + 1
                while end_idx < len(x) and res_series.iloc[end_idx] == current_res:
                    end_idx += 1
                ax.plot(x[start_idx:end_idx], y[start_idx:end_idx],
                        color=RES_COLOR_MAP.get(current_res, "black"),
                        linewidth=1.5,
                        alpha=0.9 if current_res != "year" else 0.6)
                start_idx = end_idx

        # -------------------------
        # Plot daily + cumulative in subplots
        # -------------------------
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        plot_colored_line(ax1, daily_df["date"], daily_df["daily_cases"], daily_df["T_res"])
        ax1.set_ylabel("Daily Cases")
        ax1.set_title(f"Dengue Cases in {country} (2000–present)")

        plot_colored_line(ax2, daily_df["date"], daily_df["cumulative_cases"], daily_df["T_res"])
        ax2.set_ylabel("Cumulative Cases")
        ax2.set_xlabel("Date")

        # Build a shared legend
        handles = [plt.Line2D([0], [0], color=color, lw=2, label=res.capitalize())
                   for res, color in RES_COLOR_MAP.items() if res in daily_df["T_res"].values]
        if handles:
            fig.legend(handles=handles, title="Time Resolution", loc="upper center", ncol=len(handles))

        fig.tight_layout(rect=[0, 0, 1, 0.95])
        fig.savefig(output_dir / f"{country}_dengue_plot.png", dpi=300)
        plt.close(fig)


In [ ]:
plot_all_countries(
    df=global_dengue_national,
    output_dir="figures/dengue_national"
)
